# Part 2

### Config & schema creation

In [0]:
CATALOG = "de_assessment_dev"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

In [0]:
# Imports
import time
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, ArrayType
)




In [0]:
bronze_shows = spark.table(f"{CATALOG}.bronze.bronze_shows")

rating_schema  = StructType([StructField("average", DoubleType())])
network_schema = StructType([
    StructField("id",      IntegerType()),
    StructField("name",    StringType()),
    StructField("country", StructType([StructField("name", StringType())]))
])
genres_schema = ArrayType(StringType())

silver_shows = (
    bronze_shows
    .withColumn("_rating",  F.from_json("rating",  rating_schema))
    .withColumn("_network", F.from_json("network", network_schema))
    .withColumn("_genres",  F.from_json("genres",  genres_schema))
    .select(
        F.col("id").cast(IntegerType()).alias("show_id"),
        F.col("name").alias("show_name"),
        F.col("language"),
        F.col("status"),
        F.col("runtime").cast(IntegerType()),
        F.col("averageRuntime").cast(IntegerType()).alias("average_runtime"),
        F.col("premiered"),
        F.col("ended"),
        F.col("_rating.average").alias("rating_average"),
        F.col("_network.name").alias("network_name"),
        F.col("_network.country.name").alias("network_country"),
        F.col("_genres").alias("genres"),
    )
    .withColumn("genre", F.explode_outer("genres"))
    .drop("genres")
    .filter(F.col("show_id").isNotNull())
    .fillna({"language": "Unknown", "status": "Unknown",
             "network_name": "Unknown", "network_country": "Unknown", "genre": "Unknown"})
    .dropDuplicates(["show_id", "genre"])
)

silver_shows.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.silver.silver_shows")

spark.sql(f"ALTER TABLE {CATALOG}.silver.silver_shows OWNER TO `DE-Dev-Team`")
print("silver_shows:", silver_shows.count(), "rows")